# Generate synthetic biomarker data based on ADNI

Neil Oxtoby, June 2023

1. Put ADNIMERGE CSV file in the current directory and change name accordingly (see next cell).
2. Run this. Optionally tweak which columns (biomarkers) you want.
3. Enjoy your synthetic biomarker data

In [1]:
csv_adnimerge = "ADNIMERGE_22May2023.csv"

In [2]:
# Note that I divide Ventricles by ICV to adjust approximately for head size
print("Be warned if you change biomarkers: suggest dividing brain volumes by ICV.")
biomarkers         = ["ABETA","TAU","ADAS13","MMSE","Ventricles_ICV"]
direction_abnormal = [-1,1,1,-1,1]

Be warned if you change biomarkers: suggest dividing brain volumes by ICV.


In [3]:
import pandas as pd, numpy as np
from matplotlib import pyplot as plt
import seaborn as sn

In [4]:
from scipy import stats
from pathlib import Path
import statsmodels.api as sm

In [5]:
# Approximately samples from a multivariate KDE using a multivariate normal
def resample(kde, size):
    n, d = kde.data.shape
    indices = np.random.randint(0, n, size)
    cov = np.diag(kde.bw)**2
    means = kde.data[indices, :]
    norm = np.random.multivariate_normal(np.zeros(d), cov, size)
    return np.transpose(means + norm)

In [6]:
if Path(csv_adnimerge).exists():
    df = pd.read_csv(csv_adnimerge,low_memory=False)
    df["Ventricles_ICV"] = df["Ventricles"]/df["ICV"]
    # Fix CSF strings
    for b in ['ABETA','TAU','PTAU']:
        if df[b].dtype =='O':
            # Roche Assay has a ceiling
            # OPTION 1: replace values above/below ceiling/floor with the ceiling/floor value: can result in 
            #replace_gtrless_with_me = ''
            #df[b] = df[b].str.replace('>',replace_gtrless_with_me).values
            #df[b] = df[b].str.replace('<',replace_gtrless_with_me).values
            # OPTION 2: replace values with NaN
            gtrless = df[b].str.contains('>')
            gtrless = gtrless | df[b].str.contains('<')
            df.loc[gtrless,b] = 'NaN'
            df.loc[gtrless,b] = 'NaN'
        df[b] = df[b].astype(float)
    # Baseline visit only
    bl = (df['VISCODE']=='bl').values
    # Patients and controls
    cn = (df['DX_bl']=='CN').values
    ad = (df['DX_bl']=='AD').values
    mci = (df['DX_bl'].str.contains('MCI')).astype('bool').values
    # Missing data
    missing = df[biomarkers].isnull()
    # Complete bl data
    df_complete = df.loc[ bl & (cn | ad | mci) & (missing.sum(axis=1) == 0) ].copy()
    # Fit a multivariate and sample some data from it
    #kde = sm.nonparametric.KDEMultivariate(data=X, var_type='ccccc', bw='normal_reference')
    #X_synth = resample(kde,X.shape[0]).T
    X = df_complete[biomarkers].values
    y = df_complete['DX_bl'].map({'CN':0,'AD':1,'EMCI':2,'LMCI':2}).values
    X_0 = X[y==0,:]
    X_1 = X[y==1,:]
    X_2 = X[y==2,:]
    kde_0 = sm.nonparametric.KDEMultivariate(data=X_0, var_type='ccccc', bw='normal_reference')
    X_synth_0 = resample(kde_0,X_0.shape[0]).T
    kde_1 = sm.nonparametric.KDEMultivariate(data=X_1, var_type='ccccc', bw='normal_reference')
    X_synth_1 = resample(kde_1,X_1.shape[0]).T
    kde_2 = sm.nonparametric.KDEMultivariate(data=X_2, var_type='ccccc', bw='normal_reference')
    X_synth_2 = resample(kde_2,X_2.shape[0]).T
    X_synth = np.concatenate( (X_synth_0,X_synth_1,X_synth_2) , axis=0)
    y_synth = np.concatenate( (np.zeros((X_synth_0.shape[0],)),np.ones(X_synth_1.shape[0],),2*np.ones(X_synth_2.shape[0],)) , axis=0)
    # Fix out-of-range values
    for b,k in zip(biomarkers,range(len(biomarkers))):
        mi,ma = np.nanmin(df_complete[b]),np.nanmax(df_complete[b])
        _mi = X_synth[:,k]<mi
        _ma = X_synth[:,k]>ma
        noize = np.abs(np.random.normal(0,np.abs(ma-mi)/12,sum(_mi)))
        X_synth[_mi,k] = mi + noize
        noize = np.abs(np.random.normal(0,np.abs(ma-mi)/12,sum(_ma)))
        X_synth[_ma,k] = ma - noize
    # Fix discrete-valued
    discrete = np.all(X == np.around(X),axis=0)
    X_synth[:,discrete] = np.around(X_synth[:,discrete])
    # Randomise order
    rand = np.random.permutation(X_synth.shape[0])
    X_synth = X_synth[rand,:]
    y_synth = y_synth[rand,]
    # df_stats = pd.DataFrame()
    # df_stats['biomarker'] = biomarkers
    # avg = lambda x: np.median(x)
    # spread = lambda x: stats.median_abs_deviation(x)
    # for b in biomarkers:
    #     rowz = (b==df_stats['biomarker']).values
    #     df_stats.loc[rowz,['median','mad']] = [avg(df_complete[b]),spread(df_complete[b])]        
else:
    print(f'Place ADNIMERGE CSV in current folder, update the variable csv_adnimerge and rerun. \nCurrent folder is: \n  {Path().absolute()}')


In [7]:
df_synth = pd.DataFrame(data=X_synth,columns=biomarkers)
df_synth['DX'] = y_synth
df_synth['ID'] = df_synth.index.values

In [8]:
df_synth[['ID','DX']+biomarkers].to_csv("ADNIMERGE_synthetic.csv",index=False)